# 06 — RQ6: Retrieval metrics thuần

**Câu hỏi:** Retriever (hybrid BM25 + dense + RRF) mạnh đến đâu khi không chấm câu trả lời?

**Metric:** Recall@k, MRR, nDCG@10.

In [1]:
# --- Setup ---
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
for p in [HERE] + list(HERE.parents):
    if (p / "source").is_dir() and (p / "research").is_dir():
        TRAFFIC_RAG = p
        break
else:
    raise RuntimeError("Could not locate traffic_rag/ root")

sys.path.insert(0, str(TRAFFIC_RAG))
sys.path.insert(0, str(TRAFFIC_RAG.parent))

from research.utils.langsmith_setup import enable_tracing
enable_tracing(project="traffic-rag-research", run_name="rq6-retrieval")

EVAL_PATH = TRAFFIC_RAG / "research" / "data" / "eval_qa.jsonl"
RESULTS_DIR = TRAFFIC_RAG / "research" / "results" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Chạy retriever qua eval set

In [2]:
from source.rag_core import TrafficHybridRetriever
from research.utils.eval_runner import load_eval_set
from research.utils.metrics import recall_at_k, mrr, ndcg_at_k

retriever = TrafficHybridRetriever()
eval_rows = load_eval_set(EVAL_PATH)

results = []
ks = [1, 3, 5, 10, 20]

for row in eval_rows:
    question = row['question']
    gold_citations = row.get('gold_citations', [])
    if not gold_citations: continue
    
    chunks = retriever.retrieve(question, top_k=20)
    retrieved_citations = [
        {
            'doc_id': c.metadata.get('doc_id'),
            'dieu': c.metadata.get('dieu'),
            'khoan': c.metadata.get('khoan'),
            'diem': c.metadata.get('diem'),
        }
        for c in chunks
    ]
    
    res = {'id': row['id'], 'category': row['category']}
    for k in ks:
        res[f'recall@{k}'] = recall_at_k(retrieved_citations, gold_citations, k=k)
    res['mrr'] = mrr(retrieved_citations, gold_citations)
    res['ndcg@10'] = ndcg_at_k(retrieved_citations, gold_citations, k=10)
    results.append(res)

df = pd.DataFrame(results)
summary = df.drop(columns=['id', 'category']).mean()
print(summary)
summary.to_csv(RESULTS_DIR / 'rq6_retrieval_summary.csv')


/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/rag_core/retriever.py:80: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  self.client = QdrantClient(host=qdrant_host, port=qdrant_port)


recall@1     0.050000
recall@3     0.050000
recall@5     0.050000
recall@10    0.100000
recall@20    0.100000
mrr          0.058333
ndcg@10      0.067810
dtype: float64
